In [1]:
# required libraries
import os
import torch
import torch.nn as nn

import math

import data.processor as preprocess

import mlmodel.mlp as model_mlp
import mlmodel.softdt as model_softdt
import mlmodel.tab_transformer as model_tabtrans
import mlmodel.simple_vae as model_simple_vae

processed_data_path = os.path.join(os.getcwd(), 'data/preprocessed/')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device = torch.device('cpu')

from attack.vae_sparsity_greedy_attack import VAEGreedySparsityAttack
import attack.run_gridsearch as run_gridsearch

# Remove the Future warning from pytorch
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
LOAD_EXIST = True

In [3]:
def pick_true_prediction(model, data):
    X_test_tensor, y_test_tensor = data

    with torch.no_grad():
        model.eval()
        test_outputs = model(X_test_tensor.to(device))
        predicted = torch.argmax(test_outputs, dim=1).float()
        prediction = (predicted == y_test_tensor.to(device))

        # Create tensors to hold true predictions' indices, inputs, and targets
        indices_of_true_predictions = []
        X_of_true_predictions = []
        y_of_true_predictions = []

        for i in range(len(prediction)):
            if prediction[i]:
                indices_of_true_predictions.append(i)
                X_of_true_predictions.append(X_test_tensor[i])
                y_of_true_predictions.append(y_test_tensor[i])

        return (
            torch.tensor(indices_of_true_predictions),
            torch.stack(X_of_true_predictions),
            torch.stack(y_of_true_predictions),
        )


In [4]:
parameter_grid_baseline = {
    '_lambda': [1], # Regularization parameter, 0.01, 0.03, 0.1, 0.3, 0.5, 0.7, 1 # 0.6, 0.7, 0.8, 0.9, 1
    'optimizer': ['adam'],  # Using Adam optimizer
    'lr': [0.1],  # Learning rates to try, 0.01, 0.03, 0.1, 0.15, 0.2, 0.25, 0.3
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'kappa': [0],  # Kappa value

    # Greedy search specific parameters
    "max_features": [None],  # Try with unlimited, 5, or 10 features
    "greedy_steps": [50]            # Number of greedy search steps
}

In [5]:
parameter_grid_search = {
    '_lambda': [1], # Regularization parameter, # 0.6, 0.7, 0.8, 0.9, 1
    'lambda_sparsity': [0.01, 0.1, 0.25, 0.5, 0.75, 1, 1.25, 2, 5, 10], # Sparsity parameter
    'optimizer': ['adam'],  # Using Adam optimizer
    'lr': [0.1],  # Learning rates to try
    'max_iter': [300],  # Maximum iterations to try
    'sample_num': [500],  # Number of samples to generate
    'kappa': [0],  # Kappa value
    'gamma': [1e-4],  # Gamma value
}



### Adult

In [6]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/adult')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([21112, 12]), y_train: torch.Size([21112])
X_val: torch.Size([3017, 12]), y_val: torch.Size([3017])
X_test: torch.Size([6033, 12]), y_test: torch.Size([6033])
The num of embedding dim for Transformer is: 10
The num of categorical features is: 7
The num of continues features is: 5
The num of total features is: 12
The num of binary features is: 1
Combined embedding dims: [(16, 4), (7, 3), (7, 3), (14, 4), (6, 3), (5, 3), (41, 7)]


/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.2.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Load VAE model

In [7]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "adult",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-3,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/adult_0.001_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(16, 4)
    (1-2): 2 x Embedding(7, 3)
    (3): Embedding(14, 4)
    (4): Embedding(6, 3)
    (5): Embedding(5, 3)
    (6): Embedding(41, 7)
  )
  (encoder): Sequential(
    (0): Linear(in_features=32, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0

Load ml model

In [8]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "adult",
    "epochs": 150,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_adult.pt


In [9]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "adult",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=2, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_adult.pt


In [10]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "adult",
    "epochs": 10,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                      num_continuous=continues_num,
                                      dim=trans_dim,
                                      depth=3,
                                      heads=4,
                                      dim_head=16,
                                      dim_out=2,
                                      ff_dropout=0.2,
                                      attn_dropout=0.2,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-3, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_adult.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [11]:
for model_str in ["MLP", ]: #"SoftDecisionTree", "TabTransformer"

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEGreedySparsityAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="adult",                         # Dataset name
        model=model_str,                               # Model name
        attack="greedy_sparsity",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 5151
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0, 'max_features': None, 'greedy_steps': 50}
Successfully loaded existing results for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0, 'max_features': None, 'greedy_steps': 50}
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0,
        "max_features": null,
        "greedy_steps": 50
    },
    "metrics": {
        "success_rate": 0.51,
        "mean_l2_distance": 0.6219787001609802,
        "mean_linf_distance": 0.30215272307395935,
        "perturbed_features_latent": 14.431372549019608,
        "perturbed_features_input": 2.4352941176470586,
        "mean_co

### phishing_url

In [12]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/phishing_url')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([8001, 86]), y_train: torch.Size([8001])
X_val: torch.Size([1143, 86]), y_val: torch.Size([1143])
X_test: torch.Size([2286, 86]), y_test: torch.Size([2286])
The num of embedding dim for Transformer is: 2
The num of categorical features is: 1
The num of continues features is: 85
The num of total features is: 86
The num of binary features is: 28
Combined embedding dims: [(3, 2)]


In [13]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "phishing_url",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 16,
    "hidden_layers": [128, 64],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}

simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
)

simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/phishing_url_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList(
    (0): Embedding(3, 2)
  )
  (encoder): Sequential(
    (0): Linear(in_features=87, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=64, out_features=16, bias=True)
  (_log_var_enc): Linear(in_features=64, out_features=16, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=16, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_f

Load ml model

In [14]:

mlp_config = {
    "model": "MLP",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [64, 32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=2,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-4, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_phishing_url.pt


In [15]:
softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "phishing_url",
    "epochs": 100,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num,
                                        output_dim=2,
                                        depth=softdt_config['depth'],
                                        lamda=softdt_config['lamda'],
                                        num_categorical=categorical_num,
                                        embedding_dims=embedding_dims,
                                        device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_phishing_url.pt


In [16]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "phishing_url",
    "epochs": 50,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=categories,
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=2,
                                         )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')

Model loaded from models/TabTransformer_phishing_url.pt


Apply attacks - MLP + SoftDT + Tab Transfomer

In [17]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]: # "SoftDecisionTree", "TabTransformer"

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEGreedySparsityAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="phishing_url",                         # Dataset name
        model=model_str,                               # Model name
        attack="greedy_sparsity",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2198
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0, 'max_features': None, 'greedy_steps': 50}
Could not load existing results: Attack folder does not exist: adversarial_examples/phishing_url/MLP/greedy_sparsity/Try_500_inputs/_lambda_1_lr_0.1_max_iter_300_maxfeatures_None
Running attack instead for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0, 'max_features': None, 'greedy_steps': 50}


Generating Adversarial Examples: 100%|██████████| 500/500 [04:11<00:00,  1.99it/s]



Attack Statistics:
Total successful adversarial examples: 490/500 (98.00%)
Initial success rate: 490/500 (98.00%)
Additional success from greedy search: 1 examples
Average features modified in successful attacks: 14.00
Adversarial examples confidence: Mean - 0.5865, Max - 0.9596, Min - 0.0000
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0,
        "max_features": null,
        "greedy_steps": 50
    },
    "metrics": {
        "success_rate": 0.98,
        "mean_l2_distance": 0.9132366180419922,
        "mean_linf_distance": 0.49386289715766907,
        "perturbed_features_latent": 15.738775510204082,
        "perturbed_features_input": 14.122448979591837,
        "mean_confidence_successful": 0.5983512308752659,
        "mean_confidence_unsuccessful": 0.008127784729003907,
        "min_confidence_unsuccessful": 4.4465065002441406e-05,
        "max_confidence_unsuc

Generating Adversarial Examples: 100%|██████████| 500/500 [17:39<00:00,  2.12s/it]



Attack Statistics:
Total successful adversarial examples: 372/500 (74.40%)
Initial success rate: 372/500 (74.40%)
Additional success from greedy search: 0 examples
Average features modified in successful attacks: 11.81
Adversarial examples confidence: Mean - 0.4352, Max - 0.9197, Min - 0.0507
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0,
        "max_features": null,
        "greedy_steps": 50
    },
    "metrics": {
        "success_rate": 0.744,
        "mean_l2_distance": 0.5954565405845642,
        "mean_linf_distance": 0.31312382221221924,
        "perturbed_features_latent": 15.35483870967742,
        "perturbed_features_input": 11.809139784946236,
        "mean_confidence_successful": 0.5657227787800053,
        "mean_confidence_unsuccessful": 0.0558346607722342,
        "min_confidence_unsuccessful": 0.050656795501708984,
        "max_confidence_unsuccess

Generating Adversarial Examples: 100%|██████████| 500/500 [22:45<00:00,  2.73s/it]


Attack Statistics:
Total successful adversarial examples: 485/500 (97.00%)
Initial success rate: 485/500 (97.00%)
Additional success from greedy search: 1 examples
Average features modified in successful attacks: 13.58
Adversarial examples confidence: Mean - 0.5893, Max - 0.9764, Min - 0.0000
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0,
        "max_features": null,
        "greedy_steps": 50
    },
    "metrics": {
        "success_rate": 0.97,
        "mean_l2_distance": 0.8762764930725098,
        "mean_linf_distance": 0.4792631268501282,
        "perturbed_features_latent": 15.538144329896907,
        "perturbed_features_input": 13.696907216494845,
        "mean_confidence_successful": 0.6060949097473904,
        "mean_confidence_unsuccessful": 0.04609028895696004,
        "min_confidence_unsuccessful": 8.344650268554688e-06,
        "max_confidence_unsucces

### PenDigit

In [18]:
# load the data from
X_train, X_val, X_test, y_train, y_val, y_test, preprocessor_state = \
    preprocess.load_and_use_saved_data('data/preprocessed/pendigit')

# feature numbers
feature_nums = preprocessor_state['processed_feature_nums']
total_num = X_train.shape[1]
continues_num = feature_nums['x_num']
categorical_num = total_num - continues_num
embedding_dims = feature_nums['embedding_dims']
categories = feature_nums['categories']
binary_num = feature_nums['binary_num']
total_categories = sum(categories)
trans_dim = math.ceil(math.sqrt(total_categories))
print(f'The num of embedding dim for Transformer is: {trans_dim}')
print(f'The num of categorical features is: {categorical_num}')
print(f'The num of continues features is: {continues_num}')
print(f'The num of total features is: {total_num}')
print(f'The num of binary features is: {binary_num}')

# List of (num_categories, embedding_dim)
embedding_dims = [(categories[i], embedding_dims[i]) for i in range(categorical_num)]
print(f"Combined embedding dims: {embedding_dims}")

Loaded data shapes:
X_train: torch.Size([7693, 16]), y_train: torch.Size([7693])
X_val: torch.Size([1100, 16]), y_val: torch.Size([1100])
X_test: torch.Size([2199, 16]), y_test: torch.Size([2199])
The num of embedding dim for Transformer is: 0
The num of categorical features is: 0
The num of continues features is: 16
The num of total features is: 16
The num of binary features is: 0
Combined embedding dims: []


Load VAE model

In [19]:
# define the vae model
vae_config = {
    "model": "VariationalAutoencoder",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "embedding_dims": embedding_dims,
    "latent_dim": 8,
    "hidden_layers": [64, 32],
    "kl_weight": 1e-2,
    "classification_weight": 1,
}



simplevae_model = model_simple_vae.SimpleVAE(
    data_name=vae_config['dataset'],
    layers=[total_num] + vae_config['hidden_layers'] + [vae_config['latent_dim']],
    num_categorical=categorical_num,
    num_binary=binary_num,
    num_numerical=continues_num,
    embedding_dims=vae_config['embedding_dims'],
    device=device,
    dropout=0.2,
    num_classes=10
)


simplevae_model.load(kl=vae_config['kl_weight'], epoch=vae_config['epochs'], cls_weight=vae_config['classification_weight'])

Model loaded from models/pendigit_0.01_200_1.pt


SimpleVAE(
  (embedding_layers): ModuleList()
  (encoder): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
  )
  (_mu_enc): Linear(in_features=32, out_features=8, bias=True)
  (_log_var_enc): Linear(in_features=32, out_features=8, bias=True)
  (shared_decoder): Sequential(
    (0): Linear(in_features=8, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=64, bias=True)
    (3): ReLU()
  )
  (mu_dec): Sequential(
    (0): Sequential(
      (0): Linear(in_features=8, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=64, bias=True)
      (3): Re

Load ml model

In [20]:
import mlmodel.mlp as model_mlp

mlp_config = {
    "model": "MLP",
    "dataset": "pendigit",
    "epochs": 200,
    "batch_size": 512,
    "device": device,
    "hidden_dims": [32, 16],
}

mlp = model_mlp.MLP(input_dim=total_num, 
                    hidden_dims=mlp_config['hidden_dims'],
                    output_dim=10,
                    num_categorical=categorical_num,
                    embedding_dims=embedding_dims,
                    dropout=0.2
                    ).to(mlp_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)

mlp = model_mlp.load(mlp, mlp_config['model'], mlp_config['dataset'], device=device, save_dir='models')

Model loaded from models/MLP_pendigit.pt


In [21]:
import mlmodel.softdt as model_softdt

softdt_config = {
    "model": "SoftDecisionTree",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device,
    "depth": 5,
    "lamda": 0.01
}

softdt = model_softdt.SoftDecisionTree(input_dim=total_num, 
                                       output_dim=10, 
                                       depth=5, 
                                       lamda=0.01, 
                                       num_categorical=categorical_num,
                                       embedding_dims=embedding_dims,
                                       device=device).to(softdt_config['device'])

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(softdt.parameters(), lr=1e-3, weight_decay=1e-4)

softdt = model_softdt.load(softdt, softdt_config['model'], softdt_config['dataset'], device=device, save_dir='models')

Model loaded from models/SoftDecisionTree_pendigit.pt


In [22]:
tabtran_config = {
    "model": "TabTransformer",
    "dataset": "pendigit",
    "epochs": 250,
    "batch_size": 512,
    "device": device
}

tabtrans = model_tabtrans.TabTransformer(categories=[],
                                         num_continuous=continues_num,
                                         dim=trans_dim,
                                         depth=3,
                                         heads=4,
                                         dim_head=8,
                                         dim_out=10,
                                      )

# train the model with BCELoss and Adam optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(tabtrans.parameters(), lr=1e-4, weight_decay=1e-4)

tabtrans = model_tabtrans.load(tabtrans, tabtran_config['model'], tabtran_config['dataset'], device=device, save_dir='models')


Model loaded from models/TabTransformer_pendigit.pt


/opt/homebrew/Caskroom/miniforge/base/envs/VAE-Attack/lib/python3.11/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Apply attacks - MLP + SoftDT + Tab Transfomer

In [23]:
for model_str in ["MLP", "SoftDecisionTree", "TabTransformer"]: # "SoftDecisionTree", "TabTransformer"

    if model_str == "MLP":
        model = mlp
    elif model_str == "SoftDecisionTree":
        model = softdt
    elif model_str == "TabTransformer":
        model = tabtrans
    else:
        raise ValueError("Invalid model name")
    
    print(f"Attacking {model_str} model....")

    # use true prediction to attack
    true_prediction_indices, true_prediction_X_tensor, true_prediction_y_tensor = \
        pick_true_prediction(model, (X_test, y_test))
    print("Indices of true predictions:", len(true_prediction_indices))
    sampled_true_prediction_X_tensor, sampled_prediction_y_tensor = run_gridsearch.sample_data_equal_class(true_prediction_X_tensor,
                                                                                labels=y_test[true_prediction_indices],
                                                                                n_samples=parameter_grid_baseline['sample_num'][0])
    
    # Instantiate the GridSearch
    grid_search = run_gridsearch.GridSearch(
        attack_type=VAEGreedySparsityAttack,                     # Attack type
        ml_model=model,                                # Your machine learning model
        data="pendigit",                         # Dataset name
        model=model_str,                               # Model name
        attack="greedy_sparsity",                    # Attack name
        vae_model=simplevae_model,                      # Pre-trained VAE model
        factuals=sampled_true_prediction_X_tensor,  # Instances to attack
        train_data=X_train,            # Training data
        parameter_grid=parameter_grid_baseline,            # Parameter grid
        evaluation_metric=run_gridsearch.evaluation_metric,      # Custom evaluation metric
        batch_size=128,
        device=device,                             # Specify the computation device
        load_existing=LOAD_EXIST,                     # Load existing results
    )

    # Execute the grid search
    best_params, best_metrics = grid_search.run()

    # Print the results
    print("Best Parameters:", best_params)
    print("Best Metrics:", best_metrics)
    print()

Attacking MLP model....
Indices of true predictions: 2154
------------------------------------------------------------
Running combination 1/1

Testing parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0, 'max_features': None, 'greedy_steps': 50}
Could not load existing results: Attack folder does not exist: adversarial_examples/pendigit/MLP/greedy_sparsity/Try_500_inputs/_lambda_1_lr_0.1_max_iter_300_maxfeatures_None
Running attack instead for parameters: {'_lambda': 1, 'optimizer': 'adam', 'lr': 0.1, 'max_iter': 300, 'sample_num': 500, 'kappa': 0, 'max_features': None, 'greedy_steps': 50}


Generating Adversarial Examples: 100%|██████████| 500/500 [02:23<00:00,  3.49it/s]



Attack Statistics:
Total successful adversarial examples: 454/500 (90.80%)
Initial success rate: 454/500 (90.80%)
Additional success from greedy search: 0 examples
Average features modified in successful attacks: 8.70
Adversarial examples confidence: Mean - 0.5703, Max - 0.9991, Min - 0.0001
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0,
        "max_features": null,
        "greedy_steps": 50
    },
    "metrics": {
        "success_rate": 0.908,
        "mean_l2_distance": 1.1227848529815674,
        "mean_linf_distance": 0.6939319968223572,
        "perturbed_features_latent": 7.927312775330397,
        "perturbed_features_input": 8.700440528634362,
        "mean_confidence_successful": 0.6244955546811627,
        "mean_confidence_unsuccessful": 0.035403650739918587,
        "min_confidence_unsuccessful": 9.691715240478516e-05,
        "max_confidence_unsuccess

Generating Adversarial Examples: 100%|██████████| 500/500 [14:44<00:00,  1.77s/it]



Attack Statistics:
Total successful adversarial examples: 269/500 (53.80%)
Initial success rate: 269/500 (53.80%)
Additional success from greedy search: 0 examples
Average features modified in successful attacks: 8.58
Adversarial examples confidence: Mean - 0.4085, Max - 0.9392, Min - 0.0333
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0,
        "max_features": null,
        "greedy_steps": 50
    },
    "metrics": {
        "success_rate": 0.538,
        "mean_l2_distance": 0.7817790508270264,
        "mean_linf_distance": 0.5123060941696167,
        "perturbed_features_latent": 7.881040892193308,
        "perturbed_features_input": 8.58364312267658,
        "mean_confidence_successful": 0.7029417037659197,
        "mean_confidence_unsuccessful": 0.06570588329653719,
        "min_confidence_unsuccessful": 0.03334987163543701,
        "max_confidence_unsuccessful"

Generating Adversarial Examples: 100%|██████████| 500/500 [02:35<00:00,  3.21it/s]


Attack Statistics:
Total successful adversarial examples: 487/500 (97.40%)
Initial success rate: 487/500 (97.40%)
Additional success from greedy search: 0 examples
Average features modified in successful attacks: 8.77
Adversarial examples confidence: Mean - 0.5624, Max - 0.9931, Min - 0.0024
Results: 
{
    "params": {
        "_lambda": 1,
        "optimizer": "adam",
        "lr": 0.1,
        "max_iter": 300,
        "sample_num": 500,
        "kappa": 0,
        "max_features": null,
        "greedy_steps": 50
    },
    "metrics": {
        "success_rate": 0.974,
        "mean_l2_distance": 1.0851284265518188,
        "mean_linf_distance": 0.686758816242218,
        "perturbed_features_latent": 8.0,
        "perturbed_features_input": 8.770020533880903,
        "mean_confidence_successful": 0.5769145192255245,
        "mean_confidence_unsuccessful": 0.019179802674513597,
        "min_confidence_unsuccessful": 0.002392411231994629,
        "max_confidence_unsuccessful": 0.04860800